# MovieNet-318 — BaSSL Inference (Checkpoint Only)

Load the **shipped finetuned checkpoint** and run val/test evaluation **without** SSL pretrain, finetune, or the 50 GB keyframe download.

| Artifact | Source |
|----------|--------|
| Finetuned weights | `checkpoints/bassl/bassl_finetuned.pt` (in repo) |
| Precomputed embeddings | One-time HF download (~3 GB, see below) |
| Labels & split | `data/scene318/` (in repo) |

**Fast path:** `DOWNLOAD_EMBEDDINGS = True`, `DOWNLOAD_KEYFRAMES = False`.

**Optional visualization** needs keyframes for one demo movie.

## Environment setup

Same repo discovery as `test_scene_seg_bassl.ipynb`.

In [ ]:
# --- Environment setup (Colab or local) ---
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get(
    "SCENE_SEG_REPO_URL",
    "https://github.com/lwan1/movienet-scene-segmentation.git",
)
REPO_DIR = Path(os.environ.get("SCENE_SEG_REPO_DIR", "/content/movienet-scene-segmentation"))

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not REPO_DIR.exists():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "data" / "scene318" / "meta" / "split318.json").exists():
            REPO_DIR = candidate
            break
    else:
        raise FileNotFoundError("Clone the repo or set SCENE_SEG_REPO_DIR.")

DATA_ROOT = REPO_DIR / "data"
CKPT_DIR = REPO_DIR / "checkpoints" / "bassl"
EMB_ROOT = DATA_ROOT / "embeddings" / "visual"
RESULTS_DIR = REPO_DIR / "outputs" / "results"
KEYFRAME_DIR = DATA_ROOT / "keyframes_240p"

for d in (EMB_ROOT, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

INFERENCE_CFG = json.loads((CKPT_DIR / "inference_config.json").read_text(encoding="utf-8"))
print("REPO_DIR:", REPO_DIR.resolve())
print("Checkpoint:", CKPT_DIR / INFERENCE_CFG["checkpoint_file"])


## Install dependencies

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)


## Download precomputed CLIP embeddings

Skips the 50 GB keyframe archive for evaluation. Dataset: [asmith06/scene-segmentation-embeddings](https://huggingface.co/datasets/asmith06/scene-segmentation-embeddings).

In [ ]:
# --- Inference toggles ---
DOWNLOAD_EMBEDDINGS = True
DOWNLOAD_KEYFRAMES = False
LIMIT_PER_SPLIT = None

HF_EMBEDDINGS_REPO = INFERENCE_CFG.get("hf_embeddings_repo", "asmith06/scene-segmentation-embeddings")


def embeddings_ready(movie_ids):
    return all((EMB_ROOT / f"{mid}.npz").exists() for mid in movie_ids)


def download_embeddings_hf(movie_ids=None):
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "download_embeddings_hf.py"),
        "--local-dir",
        str(DATA_ROOT / "embeddings"),
        "--repo",
        HF_EMBEDDINGS_REPO,
        "--modalities",
        "visual",
    ]
    if movie_ids:
        cmd.extend(["--movie-ids", *movie_ids])
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


if DOWNLOAD_KEYFRAMES and not KEYFRAME_DIR.exists():
    subprocess.run(
        [sys.executable, str(REPO_DIR / "scripts" / "setup_keyframes.py")],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("Skipping keyframe download (inference-only path).")


## Imports & constants

In [ ]:
import math
import re
import warnings
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Video, display
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

LABEL_POLICY = INFERENCE_CFG.get("label_policy", "lgss")
CLIP_DIM = INFERENCE_CFG["clip_dim"]
MAX_SEQ_LEN = INFERENCE_CFG["max_seq_len"]
D_MODEL = INFERENCE_CFG["d_model"]
N_HEAD = INFERENCE_CFG["n_head"]
N_LAYERS = INFERENCE_CFG["n_layers"]
DROPOUT = INFERENCE_CFG["dropout"]
FINETUNE_CKPT = CKPT_DIR / INFERENCE_CFG["checkpoint_file"]

SCENE318_DIR = DATA_ROOT / "scene318"
LABEL_DIR = SCENE318_DIR / "label318"
SPLIT_PATH = SCENE318_DIR / "meta" / "split318.json"

SOTA_REFERENCE = pd.DataFrame([
    {"method": "BaSSL (published)", "f1": 47.0, "ap": 57.4},
    {"method": "TranS4mer", "f1": 48.4, "ap": 60.8},
    {"method": "MEGA", "f1": 55.3, "ap": 58.6},
    {"method": "Scene-VLM", "f1": 62.1, "ap": 66.8},
])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Finetune checkpoint exists:", FINETUNE_CKPT.exists())


## Load split & embeddings

In [ ]:
def load_split():
    with open(SPLIT_PATH, encoding="utf-8") as f:
        split = json.load(f)
    return {k: list(split[k]) for k in ("train", "val", "test")}


def parse_label_file(label_path, policy=LABEL_POLICY):
    rows = []
    with open(label_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = re.split(r"\s+", line)
            shot_id, raw = int(parts[0]), int(parts[1])
            binary = 1 if raw == -1 else raw if policy == "lgss" else raw
            rows.append({"shot_id": shot_id, "is_boundary": int(binary)})
    return rows


split = load_split()
if LIMIT_PER_SPLIT is not None:
    split = {k: v[:LIMIT_PER_SPLIT] for k, v in split.items()}

EVAL_IDS = split["val"] + split["test"]
print(f"Eval movies: {len(EVAL_IDS)}")

if DOWNLOAD_EMBEDDINGS and not embeddings_ready(EVAL_IDS):
    subset = EVAL_IDS if len(EVAL_IDS) < 128 else None
    download_embeddings_hf(subset)

missing = [mid for mid in EVAL_IDS if not (EMB_ROOT / f"{mid}.npz").exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} embeddings in {EMB_ROOT}. "
        "Run scripts/download_embeddings_hf.py or set DOWNLOAD_EMBEDDINGS=True."
    )
print("All eval embeddings present.")


## Model & checkpoint

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class BaSSLModel(nn.Module):
    def __init__(self, input_dim=CLIP_DIM, d_model=D_MODEL, nhead=N_HEAD, nlayers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model))
        self.pos = PositionalEncoding(d_model, max_len=512)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.ssm_head = nn.Linear(d_model, 2)
        self.pp_head = nn.Linear(d_model, 1)
        self.msm_head = nn.Linear(d_model, input_dim)
        self.boundary_head = nn.Linear(d_model, 1)

    def encode(self, x, key_padding_mask=None):
        h = self.input_proj(x)
        h = self.pos(h)
        return self.encoder(h, src_key_padding_mask=key_padding_mask)

    def forward_boundary(self, x, key_padding_mask=None):
        return self.boundary_head(self.encode(x, key_padding_mask=key_padding_mask)).squeeze(-1)


def load_movie_embeddings(movie_id: str) -> np.ndarray:
    return np.load(EMB_ROOT / f"{movie_id}.npz")["X"].astype(np.float32)


model = BaSSLModel().to(DEVICE)
if not FINETUNE_CKPT.exists():
    raise FileNotFoundError(f"Missing checkpoint: {FINETUNE_CKPT}")
ckpt = torch.load(FINETUNE_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print("Loaded:", FINETUNE_CKPT)


## Inference & evaluation

In [ ]:
@torch.no_grad()
def predict_movie_scores(movie_id: str) -> np.ndarray:
    X = load_movie_embeddings(movie_id)
    T = len(X)
    if T <= MAX_SEQ_LEN:
        feats = torch.from_numpy(X).unsqueeze(0).to(DEVICE)
        return torch.sigmoid(model.forward_boundary(feats)[0]).cpu().numpy()
    scores = np.zeros(T, dtype=np.float32)
    counts = np.zeros(T, dtype=np.float32)
    stride = MAX_SEQ_LEN // 2
    for start in range(0, T - MAX_SEQ_LEN + 1, stride):
        sl = slice(start, start + MAX_SEQ_LEN)
        feats = torch.from_numpy(X[sl]).unsqueeze(0).to(DEVICE)
        s = torch.sigmoid(model.forward_boundary(feats)[0]).cpu().numpy()
        scores[sl] += s
        counts[sl] += 1
    if (T - MAX_SEQ_LEN) % stride != 0:
        sl = slice(T - MAX_SEQ_LEN, T)
        feats = torch.from_numpy(X[sl]).unsqueeze(0).to(DEVICE)
        s = torch.sigmoid(model.forward_boundary(feats)[0]).cpu().numpy()
        scores[sl] += s
        counts[sl] += 1
    return scores / np.maximum(counts, 1.0)


def eval_split(movie_ids, threshold: Optional[float] = None):
    f1s, aps, all_y, all_s = [], [], [], []
    for mid in movie_ids:
        y = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{mid}.txt")], dtype=int)
        scores = predict_movie_scores(mid)
        n = min(len(y), len(scores))
        y, scores = y[:n], scores[:n]
        if y.sum() == 0:
            continue
        aps.append(average_precision_score(y, scores))
        all_y.append(y)
        all_s.append(scores)
        if threshold is not None:
            f1s.append(f1_score(y, (scores >= threshold).astype(int), zero_division=0))
    if threshold is None and all_y:
        y_cat = np.concatenate(all_y)
        s_cat = np.concatenate(all_s)
        prec, rec, thr = precision_recall_curve(y_cat, s_cat)
        f1_arr = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
        threshold = float(thr[int(np.nanargmax(f1_arr))]) if len(f1_arr) else 0.5
        for y, scores in zip(all_y, all_s):
            f1s.append(f1_score(y, (scores >= threshold).astype(int), zero_division=0))
    return {"f1": float(np.mean(f1s)) if f1s else float("nan"), "ap": float(np.mean(aps)) if aps else float("nan"), "threshold": threshold}


def to_pct(x):
    return round(float(x) * 100, 2)


val_metrics = eval_split(split["val"], threshold=None)
test_metrics = eval_split(split["test"], threshold=val_metrics["threshold"])
print(f"VAL  | F1={to_pct(val_metrics['f1']):.2f}% AP={to_pct(val_metrics['ap']):.2f}% thr={val_metrics['threshold']:.4f}")
print(f"TEST | F1={to_pct(test_metrics['f1']):.2f}% AP={to_pct(test_metrics['ap']):.2f}%")

results = pd.DataFrame([
    {"method": "BaSSL-inspired (checkpoint inference)", "split": "val", "f1": to_pct(val_metrics["f1"]), "ap": to_pct(val_metrics["ap"]), "threshold": val_metrics["threshold"]},
    {"method": "BaSSL-inspired (checkpoint inference)", "split": "test", "f1": to_pct(test_metrics["f1"]), "ap": to_pct(test_metrics["ap"]), "threshold": val_metrics["threshold"]},
])
results.to_csv(RESULTS_DIR / "bassl_inference_results.csv", index=False)
display(results)
display(SOTA_REFERENCE)


## Optional visualization (requires keyframes)

In [ ]:
RUN_VISUALIZATION = DOWNLOAD_KEYFRAMES and KEYFRAME_DIR.exists()
DEMO_MOVIE_ID = split["test"][0]
THRESHOLD = val_metrics["threshold"]

if not RUN_VISUALIZATION:
    print("Visualization skipped. Set DOWNLOAD_KEYFRAMES=True to enable.")
else:
    y_true = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{DEMO_MOVIE_ID}.txt")], dtype=int)
    scores = predict_movie_scores(DEMO_MOVIE_ID)
    pred = (scores >= THRESHOLD).astype(int)
    n = min(len(y_true), len(scores), len(pred))
    y_true, pred = y_true[:n], pred[:n]
    print(f"Demo {DEMO_MOVIE_ID}: GT={y_true.sum()} pred={pred.sum()}")

    def find_movie_keyframe_dir(movie_id):
        for candidate in (KEYFRAME_DIR / movie_id, KEYFRAME_DIR / "240P_frames" / movie_id):
            if candidate.exists():
                return candidate
        return None

    def load_shot_keyframe(movie_id, shot_idx):
        kf_dir = find_movie_keyframe_dir(movie_id)
        p = kf_dir / f"shot_{shot_idx:04d}_img_1.jpg"
        if not p.exists():
            p = kf_dir / f"shot_{shot_idx:04d}_img_0.jpg"
        return np.array(Image.open(p).convert("RGB")) if p and p.exists() else np.zeros((135, 240, 3), dtype=np.uint8)

    def boundaries_to_scene_ids(flags):
        scene_ids = np.zeros(len(flags), dtype=int)
        sid = 0
        for i, b in enumerate(flags):
            scene_ids[i] = sid
            if b == 1:
                sid += 1
        return scene_ids

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    for ax, flags, title in zip(axes, [y_true, pred], ["Ground truth", "Predicted"]):
        scene_ids = boundaries_to_scene_ids(flags[:40])
        x = 0
        for shot_idx in range(min(40, len(flags))):
            ax.imshow(load_shot_keyframe(DEMO_MOVIE_ID, shot_idx), extent=(x, x + 1, 0, 1))
            x += 1
        ax.set_title(f"{title} — {DEMO_MOVIE_ID}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    import imageio.v3 as iio
    frames = []
    scene_ids = boundaries_to_scene_ids(pred[:200])
    prev = None
    for shot_idx in range(min(200, len(pred))):
        sid = int(scene_ids[shot_idx])
        is_new = prev is not None and sid != prev
        prev = sid
        img = Image.fromarray(load_shot_keyframe(DEMO_MOVIE_ID, shot_idx))
        draw = ImageDraw.Draw(img)
        color = (255, 64, 64) if is_new else (40, 40, 40)
        draw.rectangle([0, 0, img.width, 28], fill=color)
        draw.text((8, 6), f"Scene {sid} | shot {shot_idx:04d}", fill=(255, 255, 255))
        frames.append(np.array(img))
    mp4_path = RESULTS_DIR / f"pred_scenes_{DEMO_MOVIE_ID}.mp4"
    iio.imwrite(mp4_path, frames, fps=4, codec="libx264", quality=8, macro_block_size=1)
    display(Video(str(mp4_path), embed=True, width=640))
